# Bài 5.6 · Image segmentation: nhãn cho từng điểm ảnh

**Câu hỏi của buổi này:** khi phải nói điểm ảnh nào là người, đo đúng sai bằng gì, và vì sao U-Net cần những đường nối tắt (skip)?

Dữ liệu là **Penn-Fudan**: 170 ảnh đường phố, mỗi người đi bộ có một mask do người vẽ (bài 5.5 đã dùng box của nó). Ba model: một **U-Net nhỏ** tự train, **cùng mạng ấy bỏ skip**, và **DeepLabV3** đã học sẵn trên COCO, không train thêm.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | ảnh, mask người (semantic) và mask từng người (instance) |
| 2. Accuracy đánh lừa | model đoán toàn nền vẫn được accuracy cao, Dice bằng 0 |
| 3. Dice và IoU | tính tay trên một mask nhỏ, quan hệ giữa hai số |
| 4. Train U-Net | có skip và bỏ skip: đường Dice theo epoch |
| 5. So ba model | bảng Dice/IoU, lưới ảnh dự đoán |
| 6. Chọn ngưỡng | Dice theo ngưỡng xác suất trên validation |
| 7. Ảnh khó | Dice theo cỡ người, ảnh tệ nhất |
| 8. RLE | mã hóa mask thành chuỗi để nộp bài, giải mã lại |
| 9. Đổi kích thước mask | nearest và bilinear, Dice mất đi khi thu nhỏ |
| 10. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 3 phút. Bài giảng: [ai-canuc80k · bài 5.6](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.6).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, zipfile, urllib.request
from pathlib import Path
import torch, torchvision
from torch import nn
import torch.nn.functional as F
from torchvision.io import read_image, ImageReadMode
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
S, EPOCHS = 192, 60
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Mask của Penn-Fudan ghi 0 cho nền và 1, 2, 3… cho từng người: đó là **instance** mask. Gộp mọi số khác 0 thành 1 ta có **semantic** mask "người hay nền", thứ notebook này học. Mọi ảnh được đưa về 192 × 192 (chia hết cho 16, đủ cho bốn lần pooling); 120 ảnh train, 50 ảnh validation.

In [ ]:
root = Path('/tmp/pennfudan')
if not (root / 'PennFudanPed').exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve('https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip', root / 'data.zip')
    zipfile.ZipFile(root / 'data.zip').extractall(root)
paths = sorted((root / 'PennFudanPed/PNGImages').glob('*.png'))
raw = [read_image(str(p), ImageReadMode.RGB) for p in paths]
inst = [read_image(str(root / 'PennFudanPed/PedMasks' / (p.stem + '_mask.png')))[0] for p in paths]

X = torch.stack([F.interpolate(im[None].float() / 255, size=(S, S), mode='bilinear', antialias=True, align_corners=False)[0].clamp(0, 1) for im in raw])
Y = torch.stack([F.interpolate((m > 0)[None, None].float(), size=(S, S), mode='nearest')[0] for m in inst])
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
tr, va = perm[:120], perm[120:]
fg = Y.mean().item()
table(pd.DataFrame([('số ảnh', f'{len(X)} (train {len(tr)}, validation {len(va)})'), ('người mỗi ảnh', f'{min(int(m.max()) for m in inst)}–{max(int(m.max()) for m in inst)}'),
                    ('cỡ gốc', f'{min(i.shape[2] for i in raw)}–{max(i.shape[2] for i in raw)} pixel chiều ngang'), ('tỉ lệ pixel là người', f'{fg:.1%}')],
                   columns=['', 'giá trị']), 'Penn-Fudan cho segmentation')
check('mask chỉ có hai giá trị 0 và 1', set(Y.unique().tolist()) == {0.0, 1.0})

fig, axes = plt.subplots(3, 4, figsize=(14, 10.5))
for j, i in enumerate([3, 40, 90, 150]):
    axes[0, j].imshow(raw[i].permute(1, 2, 0)); axes[0, j].set_title(f'ảnh {raw[i].shape[2]} × {raw[i].shape[1]}', fontsize=10)
    axes[1, j].imshow(inst[i], cmap='tab10', vmin=0, vmax=9, interpolation='nearest'); axes[1, j].set_title(f'instance: {int(inst[i].max())} người, mỗi người một màu', fontsize=10)
    axes[2, j].imshow(Y[i, 0], cmap='gray', interpolation='nearest'); axes[2, j].set_title(f'semantic {S} × {S}: {Y[i].mean():.0%} pixel là người', fontsize=10)
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Accuracy đánh lừa

Pixel accuracy là tỉ lệ pixel đoán đúng. Khi nền chiếm phần lớn ảnh, một "model" luôn đoán nền đã được điểm cao. **Dice** và **IoU** chỉ nhìn vùng người:

$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}, \qquad \text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

với $A$ là vùng đoán, $B$ là vùng thật. Ở đây Dice và IoU được **cộng dồn trên toàn tập validation** (cộng giao, cộng hợp của mọi ảnh rồi mới chia).

In [ ]:
def scores(pred, gt):
    pred, gt = pred.bool(), gt.bool()
    inter, a, b = (pred & gt).sum().item(), pred.sum().item(), gt.sum().item()
    return {'pixel accuracy': (pred == gt).float().mean().item(),
            'Dice': 2 * inter / (a + b) if a + b else 1.0, 'IoU': inter / (a + b - inter) if a + b - inter else 1.0}

Yv = Y[va]
rows = [('luôn đoán nền', scores(torch.zeros_like(Yv), Yv)), ('luôn đoán người', scores(torch.ones_like(Yv), Yv)),
        ('đoán đúng mask, dời 8 pixel sang phải', scores(torch.roll(Yv, 8, dims=3), Yv))]
base = pd.DataFrame([{'dự đoán': n} | s for n, s in rows])
table(base, 'Ba dự đoán ngây thơ trên validation', fmt={c: '{:.3f}' for c in base.columns[1:]})
fig, ax = plt.subplots(figsize=(10, 3.4))
xs = np.arange(len(base))
for k, (col, c) in enumerate([('pixel accuracy', C['gray']), ('Dice', C['clay']), ('IoU', C['blue'])]):
    ax.bar(xs + (k - 1) * .26, base[col], .26, color=c, label=col)
bar_labels(ax, '{:.2f}'); ax.set_xticks(xs, base['dự đoán']); ax.set_ylim(0, 1.12); ax.legend(frameon=False, ncol=3, loc='upper right')
ax.set_title('Đoán toàn nền: accuracy cao, Dice bằng 0')
plt.tight_layout(); plt.show()
check('đoán toàn nền có accuracy trên 0.7 nhưng Dice bằng 0', base.iloc[0, 1] > 0.7 and base.iloc[0, 2] == 0)
note(f'Người chỉ chiếm {Yv.mean():.0%} số pixel, nên đoán toàn nền đã đúng <b>{base.iloc[0, 1]:.0%}</b> pixel mà không tìm ra người nào. '
     f'Chỉ dời mask đúng 8 pixel, accuracy vẫn {base.iloc[2, 1]:.0%}, nhưng Dice tụt còn {base.iloc[2, 2]:.2f}: Dice và IoU nhạy với biên, đúng thứ segmentation cần.')

## 3. Dice và IoU

Tính tay trên hai mask 6 × 6 rồi so với hàm `scores`. Với một cặp mask, hai số luôn đi cùng nhau theo công thức $\text{Dice} = 2\,\text{IoU} / (1 + \text{IoU})$, nên xếp hạng model theo Dice hay IoU trên **một** ảnh là như nhau.

In [ ]:
A = torch.zeros(6, 6); A[1:4, 1:5] = 1          # đoán: 3 × 4 = 12 pixel
B = torch.zeros(6, 6); B[2:5, 2:5] = 1          # thật: 3 × 3 = 9 pixel
inter = int((A * B).sum()); s = scores(A, B)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, (m, t) in zip(axes, [(A, f'đoán A: {int(A.sum())} pixel'), (B, f'thật B: {int(B.sum())} pixel'), (A + 2 * B, f'giao: {inter} pixel')]):
    ax.imshow(m, cmap='magma', vmin=0, vmax=3); ax.set_title(t, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xticks(np.arange(-.5, 6), minor=True); ax.set_yticks(np.arange(-.5, 6), minor=True); ax.grid(which='minor', color='white', lw=.6)
plt.tight_layout(); plt.show()
hand = pd.DataFrame([('Dice', f'2·{inter} / ({int(A.sum())} + {int(B.sum())})', 2 * inter / (A.sum() + B.sum()).item(), s['Dice']),
                     ('IoU', f'{inter} / ({int(A.sum())} + {int(B.sum())} − {inter})', inter / (A.sum() + B.sum() - inter).item(), s['IoU'])],
                    columns=['metric', 'tính tay', 'kết quả', 'hàm scores'])
table(hand, 'Hai mask 6 × 6', fmt={'kết quả': '{:.4f}', 'hàm scores': '{:.4f}'})
check('tính tay khớp hàm scores', abs(hand.iloc[0, 2] - s['Dice']) < 1e-9 and abs(hand.iloc[1, 2] - s['IoU']) < 1e-9)
check('Dice = 2 IoU / (1 + IoU)', abs(s['Dice'] - 2 * s['IoU'] / (1 + s['IoU'])) < 1e-9)

## 4. Train U-Net

U-Net nhỏ: bốn lần thu nhỏ (32 → 64 → 128 → 256 kênh, đáy 512 kênh ở 12 × 12), rồi bốn lần phóng to bằng `ConvTranspose2d`. Ở mỗi mức, feature map của encoder được **nối kênh** vào decoder. Bản "bỏ skip" giống hệt nhưng thay feature map của encoder bằng số 0, nên decoder chỉ còn thông tin từ đáy 12 × 12.

Loss là BCE cộng Dice loss; lật ngang ngẫu nhiên khi train; mỗi epoch đo Dice trên validation với ngưỡng 0.5.

In [ ]:
def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, skip=True, ch=(32, 64, 128, 256, 512)):
        super().__init__()
        self.skip = skip
        self.down = nn.ModuleList([block(i, o) for i, o in zip((3,) + ch[:-2], ch[:-1])])
        self.bottom = block(ch[-2], ch[-1])
        levels = list(reversed(range(len(ch) - 1)))                                    # 3, 2, 1, 0
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[k + 1], ch[k], 2, stride=2) for k in levels])
        self.dec = nn.ModuleList([block(2 * ch[k], ch[k]) for k in levels])
        self.head = nn.Conv2d(ch[0], 1, 1)
    def forward(self, x):
        skips = []
        for d in self.down:
            x = d(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.bottom(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = up(x)
            x = dec(torch.cat([x, s if self.skip else torch.zeros_like(s)], dim=1))   # nối kênh: C + C
        return self.head(x)

MEAN, STD = torch.tensor([0.485, 0.456, 0.406])[:, None, None], torch.tensor([0.229, 0.224, 0.225])[:, None, None]
Xn = (X - MEAN) / STD
def predict(m, idx):
    m.eval()
    with torch.no_grad(): return torch.cat([torch.sigmoid(m(Xn[idx[i:i + 25]].to(device))).cpu() for i in range(0, len(idx), 25)])

def train(skip):
    torch.manual_seed(0)
    m = UNet(skip).to(device); opt = torch.optim.Adam(m.parameters(), 1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    hist = []
    for ep in range(EPOCHS):
        m.train()
        for b in tr[torch.randperm(len(tr))].split(8):
            x, y = Xn[b].to(device), Y[b].to(device)
            flip = torch.rand(len(b), device=device) < .5
            x = torch.where(flip[:, None, None, None], x.flip(3), x); y = torch.where(flip[:, None, None, None], y.flip(3), y)
            logit = m(x); p = torch.sigmoid(logit)
            dice_loss = 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
            loss = F.binary_cross_entropy_with_logits(logit, y) + dice_loss
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        hist.append(scores(predict(m, va) > .5, Yv)['Dice'])
    return m, hist

t = time.time()
unet, h_skip = train(True)
unet_ns, h_noskip = train(False)
t_train = time.time() - t
n_par = sum(p.numel() for p in unet.parameters())
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(range(1, EPOCHS + 1), h_skip, color=C['clay'], lw=2, label='U-Net có skip')
ax.plot(range(1, EPOCHS + 1), h_noskip, color=C['blue'], lw=2, label='bỏ skip')
ax.set(xlabel='epoch', ylabel='Dice validation', title=f'Hai mạng {n_par / 1e6:.1f} triệu tham số, train {len(tr)} ảnh ({t_train:.0f} giây cho cả hai)'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
check('U-Net có skip vượt mức đoán toàn nền', h_skip[-1] > 0.3)
note(f'Sau {EPOCHS} epoch, Dice validation: có skip <b>{h_skip[-1]:.3f}</b>, bỏ skip <b>{h_noskip[-1]:.3f}</b>. Hai mạng có cùng số tham số; khác nhau chỉ ở chỗ decoder có được nhìn lại feature map độ phân giải cao hay không.')

## 5. So ba model

**DeepLabV3** (ResNet-50) đã học trên COCO với 21 lớp kiểu Pascal VOC, trong đó có lớp "person". Lấy xác suất lớp person, đưa về 192 × 192, không train thêm trên Penn-Fudan.

In [ ]:
W = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
with quiet(): dl = deeplabv3_resnet50(weights=W).to(device).eval()
PERSON = W.meta['categories'].index('person')
t = time.time()
dl_prob = []
with torch.no_grad():
    for i in va.tolist():
        x = W.transforms()(raw[i])[None].to(device)
        p = dl(x)['out'].softmax(1)[:, PERSON:PERSON + 1]
        dl_prob.append(F.interpolate(p, size=(S, S), mode='bilinear', align_corners=False).cpu()[0])
dl_prob = torch.stack(dl_prob); t_dl = time.time() - t
probs = {'U-Net bỏ skip': predict(unet_ns, va), 'U-Net có skip': predict(unet, va), 'DeepLabV3 (COCO, không train thêm)': dl_prob}
band = (F.max_pool2d(Yv, 9, 1, 4) - (-F.max_pool2d(-Yv, 9, 1, 4))).bool()       # pixel cách biên người không quá 4 px
res = pd.DataFrame([{'model': n} | scores(p > .5, Yv) | {'đúng trong dải biên ±4 px': ((p > .5) == Yv.bool())[band].float().mean().item()} for n, p in probs.items()])
table(res, f'Validation {len(va)} ảnh, ngưỡng 0.5 (dải biên: {band.float().mean():.0%} số pixel)', fmt={c: '{:.3f}' for c in res.columns[1:]}, best='Dice')

show = [int(k) for k in torch.argsort(Yv.mean((1, 2, 3)))[torch.tensor([5, 25, 45])]]
fig, axes = plt.subplots(3, 5, figsize=(15, 9.4))
for r, k in enumerate(show):
    axes[r, 0].imshow(X[va[k]].permute(1, 2, 0)); axes[r, 1].imshow(Yv[k, 0], cmap='gray')
    for c, (n, p) in enumerate(probs.items(), start=2):
        d = scores(p[k] > .5, Yv[k])['Dice']
        axes[r, c].imshow(p[k, 0], cmap='magma', vmin=0, vmax=1); axes[r, c].set_title(f'{n.split(" (")[0]}: Dice {d:.2f}', fontsize=10)
    axes[r, 0].set_title('ảnh', fontsize=10); axes[r, 1].set_title('mask thật', fontsize=10)
for ax in axes.flat: ax.axis('off')
fig.suptitle('Xác suất "người" từng pixel (sáng là gần 1)', y=0.995)
plt.tight_layout(); plt.show()
r_ = res.set_index('model')['Dice']; rb = res.set_index('model')['đúng trong dải biên ±4 px']
check('U-Net có skip đúng ở biên nhiều hơn bản bỏ skip', rb['U-Net có skip'] > rb['U-Net bỏ skip'])
gap, gapb = r_['U-Net có skip'] - r_['U-Net bỏ skip'], rb['U-Net có skip'] - rb['U-Net bỏ skip']
note(f'Bỏ skip, decoder chỉ còn bản đồ 12 × 12: Dice {r_["U-Net bỏ skip"]:.3f} so với {r_["U-Net có skip"]:.3f} ({gap:+.3f}); trong dải biên, tỉ lệ pixel đúng {rb["U-Net bỏ skip"]:.3f} so với {rb["U-Net có skip"]:.3f} ({gapb:+.3f}). '
     + ('Phần thân người to và liền nên mạng bỏ skip vẫn đoán được; skip giúp chủ yếu ở biên, chân tay, người nhỏ. ' if gap < 0.03 else 'Skip mang chi tiết độ phân giải cao về cho decoder, nên biên và người nhỏ rõ hơn hẳn. ')
     + f'DeepLabV3 học từ hàng chục nghìn ảnh COCO nên đạt {r_.iloc[2]:.3f} mà không cần train thêm ({t_dl:.0f} giây cho {len(va)} ảnh): với {len(tr)} ảnh train, model học sẵn là điểm xuất phát tốt hơn train từ đầu.')

## 6. Chọn ngưỡng

Mask cuối cùng là `xác suất > ngưỡng`. Ngưỡng 0.5 không có gì đặc biệt; Dice loss và lớp hiếm làm xác suất lệch. Quét ngưỡng trên validation (khi đi thi: trên một phần tách riêng, không phải test).

In [ ]:
ths = np.round(np.arange(.1, .91, .05), 2)
fig, ax = plt.subplots(figsize=(10, 3.8))
best = {}
for (n, p), c in zip(probs.items(), [C['blue'], C['clay'], C['green']]):
    d = [scores(p > t_, Yv)['Dice'] for t_ in ths]
    ax.plot(ths, d, 'o-', ms=3, color=c, label=n); best[n] = (ths[int(np.argmax(d))], max(d), d[list(ths).index(.5)])
    ax.plot(best[n][0], best[n][1], '*', ms=13, color=c)
ax.set(xlabel='ngưỡng xác suất', ylabel='Dice validation', title='Dice theo ngưỡng (sao: ngưỡng tốt nhất)'); ax.legend(frameon=False, loc='lower center')
plt.tight_layout(); plt.show()
bt = pd.DataFrame([(n, b[2], b[0], b[1], b[1] - b[2]) for n, b in best.items()], columns=['model', 'Dice ở 0.5', 'ngưỡng tốt nhất', 'Dice tốt nhất', 'tăng thêm'])
table(bt, 'Ngưỡng tốt nhất của từng model', fmt={'Dice ở 0.5': '{:.3f}', 'ngưỡng tốt nhất': '{:.2f}', 'Dice tốt nhất': '{:.3f}', 'tăng thêm': '{:+.3f}'})
note(f'Đổi ngưỡng tăng Dice nhiều nhất {bt["tăng thêm"].max():.3f} (với {bt.loc[bt["tăng thêm"].idxmax(), "model"]}): '
     + ('đáng làm nhưng không thay được một model tốt hơn.' if bt['tăng thêm'].max() < 0.03 else 'không nhỏ, nên luôn quét ngưỡng trước khi nộp.'))

## 7. Ảnh khó

Dice từng ảnh của U-Net có skip và DeepLabV3, theo phần trăm diện tích ảnh là người, rồi xem kỹ ảnh U-Net làm tệ nhất.

In [ ]:
per = pd.DataFrame({'diện tích người': Yv.mean((1, 2, 3)).numpy(),
                    'U-Net có skip': [scores(probs['U-Net có skip'][k] > .5, Yv[k])['Dice'] for k in range(len(va))],
                    'DeepLabV3': [scores(dl_prob[k] > .5, Yv[k])['Dice'] for k in range(len(va))]})
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={'width_ratios': [1.25, 1]})
for col, c in [('U-Net có skip', C['clay']), ('DeepLabV3', C['green'])]:
    axes[0].scatter(per['diện tích người'] * 100, per[col], s=22, color=c, label=col)
axes[0].set(xlabel='% diện tích ảnh là người', ylabel='Dice của ảnh', title='Mỗi chấm một ảnh validation'); axes[0].legend(frameon=False)
k = int(per['U-Net có skip'].idxmin())
over = np.zeros((S, S, 3)); pr = (probs['U-Net có skip'][k, 0] > .5).numpy(); gt = Yv[k, 0].numpy().astype(bool)
over[gt & pr] = [.30, .45, .22]; over[gt & ~pr] = [.23, .43, .66]; over[~gt & pr] = [.76, .38, .24]
axes[1].imshow(X[va[k]].permute(1, 2, 0)); axes[1].imshow(over, alpha=.65); axes[1].axis('off')
axes[1].set_title(f'Ảnh tệ nhất của U-Net (Dice {per.at[k, "U-Net có skip"]:.2f}): xanh lá đúng, xanh dương sót, cam thừa', fontsize=10)
plt.tight_layout(); plt.show()
small = per['diện tích người'] < per['diện tích người'].median()
table(pd.DataFrame([('người nhỏ (dưới trung vị)', per.loc[small, 'U-Net có skip'].mean(), per.loc[small, 'DeepLabV3'].mean()),
                    ('người lớn', per.loc[~small, 'U-Net có skip'].mean(), per.loc[~small, 'DeepLabV3'].mean())],
                   columns=['nhóm ảnh', 'U-Net có skip', 'DeepLabV3']), 'Dice trung bình theo cỡ người', fmt={'U-Net có skip': '{:.3f}', 'DeepLabV3': '{:.3f}'})
harder = {c: 'người nhỏ' if per.loc[small, c].mean() < per.loc[~small, c].mean() else 'người lớn' for c in ['U-Net có skip', 'DeepLabV3']}
note(f'Ảnh U-Net làm tệ nhất có Dice {per.at[k, "U-Net có skip"]:.2f} (DeepLabV3 {per.at[k, "DeepLabV3"]:.2f} trên cùng ảnh). '
     + (f'Cả hai model đều kém hơn ở ảnh {harder["DeepLabV3"]}. ' if harder['DeepLabV3'] == harder['U-Net có skip'] else
        f'U-Net kém hơn ở ảnh {harder["U-Net có skip"]}, DeepLabV3 kém hơn ở ảnh {harder["DeepLabV3"]}: hai model khó ở những chỗ khác nhau. ')
     + 'Điểm trung bình che mất những ảnh này; khi đi thi, xem vài ảnh tệ nhất trên validation trước khi đổi model.')

## 8. RLE: mã hóa mask để nộp bài

Một mask 192 × 192 có 36 864 số 0/1. Các cuộc thi segmentation thường nhận **run-length encoding**: duyệt pixel theo một thứ tự quy ước, ghi cặp (vị trí bắt đầu, độ dài) cho mỗi đoạn số 1 liên tiếp. Ở đây theo quy ước hay gặp trên Kaggle: duyệt **theo cột** (từ trên xuống, rồi sang cột kế), vị trí đánh số **từ 1**.

In [ ]:
def rle_encode(mask):
    flat = np.concatenate([[0], mask.T.flatten(), [0]])          # .T: duyệt theo cột
    edges = np.flatnonzero(flat[1:] != flat[:-1]) + 1              # vị trí đổi giá trị, đánh số từ 1
    edges[1::2] -= edges[::2]                                      # (bắt đầu, kết thúc) -> (bắt đầu, độ dài)
    return ' '.join(map(str, edges))

def rle_decode(s, shape):
    v = np.array(s.split(), int); flat = np.zeros(shape[0] * shape[1], np.uint8)
    for st, ln in zip(v[::2], v[1::2]): flat[st - 1:st - 1 + ln] = 1
    return flat.reshape(shape[1], shape[0]).T

tiny = np.array([[0, 1, 0], [1, 1, 0], [0, 0, 1]])
m = (probs['U-Net có skip'][0, 0] > .5).numpy().astype(np.uint8)
code = rle_encode(m)
table(pd.DataFrame([('mask 3 × 3 ở dưới', rle_encode(tiny)), (f'mask dự đoán {S} × {S} (40 ký tự đầu)', code[:40] + ' …')], columns=['mask', 'RLE']), 'Hai ví dụ')
display(HTML('<pre style="font-size:15px;line-height:1.3;margin:4px 0 10px">' + '\n'.join(' '.join(map(str, r)) for r in tiny) + '</pre>'))
sp = lambda n: f'{n:,}'.replace(',', ' ')
table(pd.DataFrame([('số pixel', sp(m.size)), ('số pixel là người', sp(int(m.sum()))), ('số đoạn', sp(len(code.split()) // 2)), ('độ dài chuỗi RLE (ký tự)', sp(len(code)))],
                   columns=['', 'giá trị']), 'Mask dự đoán của một ảnh validation')
check('giải mã RLE cho lại đúng mask 3 × 3', (rle_decode(rle_encode(tiny), tiny.shape) == tiny).all())
check('giải mã RLE cho lại đúng mask dự đoán', (rle_decode(code, m.shape) == m).all())
note(f'Mask 3 × 3 đọc theo cột là {" · ".join(" ".join(map(str, c)) for c in tiny.T)}: ba đoạn số 1, nên RLE có ba cặp "{rle_encode(tiny)}". '
     f'Mask thật {sp(m.size)} pixel chỉ cần {sp(len(code))} ký tự. Đọc sai quy ước (theo hàng hay theo cột, đánh số từ 0 hay 1) thì file nộp vẫn hợp lệ mà điểm gần 0: luôn giải mã lại và so với mask trước khi nộp.')

## 9. Đổi kích thước mask

Model dự đoán ở 192 × 192, nhưng bài thi chấm ở kích thước gốc. Hai lỗi hay gặp: phóng mask nhãn bằng **bilinear** (sinh ra giá trị lai không phải nhãn), và quên rằng thu nhỏ làm mất biên. Thử ngay với mask thật.

In [ ]:
m0 = inst[next(i for i, m_ in enumerate(inst) if m_.max() >= 2)]     # instance mask gốc có ít nhất hai người
H0, W0 = m0.shape
small_ = F.interpolate(m0[None, None].float(), size=(48, 48), mode='nearest')
up_near = F.interpolate(small_, size=(H0, W0), mode='nearest')[0, 0]
up_bil = F.interpolate(small_, size=(H0, W0), mode='bilinear', align_corners=False)[0, 0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, (mm, t_) in zip(axes, [(m0, f'gốc {W0} × {H0}: {len(m0.unique())} giá trị'), (up_near, f'48 × 48 rồi phóng bằng nearest: {len(up_near.unique())} giá trị'),
                               (up_bil, f'48 × 48 rồi phóng bằng bilinear: {len(up_bil.unique())} giá trị')]):
    ax.imshow(mm, cmap='magma', vmin=0, vmax=m0.max().item(), interpolation='nearest'); ax.set_title(t_, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()

rows = []
for s_ in [24, 48, 96, 192, 384]:
    d = []
    for m_ in inst:
        g = (m_ > 0).float()[None, None]
        back = F.interpolate(F.interpolate(g, size=(s_, s_), mode='nearest'), size=g.shape[2:], mode='nearest')
        d.append(scores(back[0, 0] > .5, g[0, 0])['Dice'])
    rows.append((f'{s_} × {s_}', np.mean(d)))
rs = pd.DataFrame(rows, columns=['thu về rồi phóng lại', 'Dice với mask gốc (trung bình 170 ảnh)'])
table(rs, 'Mask thật, không có model nào: chỉ thu nhỏ rồi phóng lại', fmt={rs.columns[1]: '{:.3f}'})
check('phóng bằng nearest giữ đúng tập nhãn', set(up_near.unique().tolist()) <= set(m0.unique().tolist()))
check('phóng bằng bilinear sinh giá trị không phải nhãn', len(up_bil.unique()) > len(m0.unique()))
note(f'Bilinear sinh ra {sp(len(up_bil.unique()))} giá trị khác nhau thay vì {len(m0.unique())}: những số lẻ như 0.5 hay 1.5 không ứng với người nào; mask nhãn luôn phóng bằng nearest (còn <b>xác suất</b> thì phóng bilinear rồi mới đặt ngưỡng, như mục 5 làm với DeepLabV3). '
     f'Ngay cả mask đúng tuyệt đối, đi qua 192 × 192 rồi về kích thước gốc cũng chỉ còn Dice {rs.iloc[3, 1]:.3f}: nếu bài thi chấm ở kích thước gốc, đó là trần của mọi model dự đoán ở 192 × 192.')

## 10. Tóm tắt

In [ ]:
summary = [('tỉ lệ pixel là người', f'{fg:.1%}'),
           ('đoán toàn nền: pixel accuracy / Dice', f'{base.iloc[0, 1]:.3f} / {base.iloc[0, 2]:.0f}'),
           ('Dice validation: bỏ skip / có skip', f'{r_["U-Net bỏ skip"]:.3f} / {r_["U-Net có skip"]:.3f}'),
           ('Dice validation: DeepLabV3 không train thêm', f'{r_.iloc[2]:.3f}'),
           ('quét ngưỡng tăng Dice nhiều nhất', f'{bt["tăng thêm"].max():+.3f}'),
           (f'RLE của một mask {S} × {S}', f'{len(code)} ký tự'),
           ('trần Dice khi đi qua 192 × 192', f'{rs.iloc[3, 1]:.3f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** bỏ lật ngang khi train và xem Dice đổi bao nhiêu; giảm `S` xuống 96 rồi xem trần ở mục 9; thay encoder của U-Net bằng ResNet-18 học sẵn để xem khoảng cách với DeepLabV3 thu hẹp tới đâu.